In [ ]:
import json
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any

import anthropic
import voyageai
from pinecone import Pinecone, PodSpec, ServerlessSpec
from tqdm import tqdm

# ── Constants ─────────────────────────────────────────────────────────────────
MODEL_NAME       = "claude-3-5-sonnet-20241022"
VOYAGE_DIMENSION = 1024   # voyage-2 output vector size

DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""

CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall
document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""


# =============================================================================
# ContextualPineconeDB
# =============================================================================

class ContextualPineconeDB:
    """
    A contextual vector database backed by Pinecone (production-grade,
    HNSW-indexed, cloud-hosted).

    For each chunk, Claude generates a short situating context which is
    prepended to the chunk before embedding. Both the original content and
    the generated context are stored as separate metadata fields so you can
    inspect what Claude produced for every chunk.

    Two index modes:
        use_pod=False  → ServerlessSpec  (cheap, scales to zero, good for dev)
        use_pod=True   → PodSpec         (dedicated hardware, production grade)

    Usage:
        db = ContextualPineconeDB(
            name              = "rag-research",
            voyage_api_key    = "...",
            anthropic_api_key = "...",
            pinecone_api_key  = "...",
            use_pod           = False,   # serverless for dev
        )
        db.load_data(dataset, parallel_threads=1)
        results = db.search("how does DPR work?", k=20)
    """

    def __init__(
        self,
        name:              str,
        voyage_api_key:    str | None = None,
        anthropic_api_key: str | None = None,
        pinecone_api_key:  str | None = None,
        namespace:         str        = "research_papers",
        use_pod:           bool       = False,
    ):
        # ── API clients ───────────────────────────────────────────────────────
        if voyage_api_key is None:
            voyage_api_key = os.getenv("VOYAGE_API_KEY")
        if anthropic_api_key is None:
            anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
        if pinecone_api_key is None:
            pinecone_api_key = os.getenv("PINECONE_API_KEY")

        self.voyage_client    = voyageai.Client(api_key=voyage_api_key)
        self.anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)
        self.name             = name
        self.namespace        = namespace

        # ── Query cache (avoids re-embedding identical queries) ───────────────
        self.query_cache: dict[str, list[float]] = {}

        # ── Token usage tracking (thread-safe) ───────────────────────────────
        self.token_counts = {
            "input":          0,
            "output":         0,
            "cache_read":     0,
            "cache_creation": 0,
        }
        self.token_lock = threading.Lock()

        # ── Pinecone client + index ───────────────────────────────────────────
        self.pc = Pinecone(api_key=pinecone_api_key)

        # Create index if it doesn't exist yet
        existing_indexes = [idx.name for idx in self.pc.list_indexes()]

        if name not in existing_indexes:
            if use_pod:
                # ── Production: dedicated pod hardware ────────────────────────
                # pod_type options:
                #   p1.x1  → storage-optimised, lower QPS
                #   p1.x2  → 2x replicas of p1
                #   p2.x1  → speed-optimised, higher QPS
                #   p2.x8  → 8x replicas of p2 (high availability)
                # pods     → horizontal scale (more pods = more capacity)
                # replicas → copies for high availability + read throughput
                # shards   → split index across shards for very large datasets
                spec = PodSpec(
                    environment = "us-east-1-aws",
                    pod_type    = "p2.x1",
                    pods        = 1,
                    replicas    = 1,
                    shards      = 1,
                )
            else:
                # ── Development: serverless (scales to zero, pay-per-use) ─────
                spec = ServerlessSpec(
                    cloud  = "aws",
                    region = "us-east-1",
                )

            self.pc.create_index(
                name      = name,
                dimension = VOYAGE_DIMENSION,   # voyage-2 = 1024 dims
                # options: "cosine" | "euclidean" | "dotproduct"
                # cosine = best for text similarity (normalises magnitude)
                metric    = "cosine",
                spec      = spec,
            )
            print(f"Created Pinecone index '{name}'.")
        else:
            print(f"Using existing Pinecone index '{name}'.")

        self.index = self.pc.Index(name)

    # =========================================================================
    # situate_context
    # =========================================================================

    def situate_context(self, doc: str, chunk: str) -> tuple[str, Any]:
        """
        Ask Claude to generate a short situating context for one chunk.

        The full document is marked with cache_control so Anthropic caches it
        server-side. Every chunk after the first in the same document reads the
        document from cache at ~10% of normal token cost.

        Returns:
            (contextualized_text, usage)
        """
        response = self.anthropic_client.messages.create(
            model      = MODEL_NAME,
            max_tokens = 1000,
            temperature= 0.0,
            messages   = [
                {
                    "role": "user",
                    "content": [
                        {
                            # ── full document (cached after first chunk) ──────
                            "type": "text",
                            "text": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc),
                            "cache_control": {"type": "ephemeral"},
                        },
                        {
                            # ── the specific chunk to situate ─────────────────
                            "type": "text",
                            "text": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk),
                        },
                    ],
                }
            ],
            extra_headers={"anthropic-beta": "prompt-caching-2024-07-31"},
        )
        return response.content[0].text, response.usage

    # =========================================================================
    # load_data
    # =========================================================================

    def load_data(
        self,
        dataset:          list[dict[str, Any]],
        parallel_threads: int = 1,
    ) -> None:
        """
        Process every chunk in the dataset:
          1. Generate situating context via Claude (with prompt caching)
          2. Embed  original + context  with Voyage AI
          3. Store vectors + metadata in Pinecone

        If the namespace already contains vectors, loading is skipped entirely
        to avoid redundant Claude / Voyage API calls.

        Args:
            dataset:          Output of DocumentProcessor.run()
            parallel_threads: Worker threads for Claude calls.
                              1  = sequential (best cache hit rate, recommended
                                   for large documents with many chunks)
                              >1 = parallel   (faster wall-clock, lower cache
                                   hit rate — use for small docs / few chunks)
        """
        # ── Guard: skip if namespace already has vectors ──────────────────────
        # describe_index_stats() returns per-namespace vector counts
        stats            = self.index.describe_index_stats()
        namespace_counts = stats.get("namespaces", {})
        existing_count   = (
            namespace_counts.get(self.namespace, {}).get("vector_count", 0)
        )

        if existing_count > 0:
            print(
                f"Namespace '{self.namespace}' already contains "
                f"{existing_count:,} vectors. Skipping load."
            )
            return

        total_chunks = sum(len(doc["chunks"]) for doc in dataset)
        print(f"Processing {total_chunks} chunks with {parallel_threads} thread(s).")

        # ── Nested worker: one call per chunk ─────────────────────────────────
        def process_chunk(doc: dict, chunk: dict) -> dict:
            """
            Situate one chunk within its source document.

            Returns a dict with:
              - text_to_embed:  what gets sent to Voyage for embedding
              - metadata:       stored in Pinecone alongside the vector
              - chunk_id:       unique Pinecone vector ID
            """
            contextualized_text, usage = self.situate_context(
                doc["content"], chunk["content"]
            )

            # ── update shared token counters thread-safely ────────────────
            with self.token_lock:
                self.token_counts["input"]          += usage.input_tokens
                self.token_counts["output"]         += usage.output_tokens
                self.token_counts["cache_read"]     += usage.cache_read_input_tokens
                self.token_counts["cache_creation"] += usage.cache_creation_input_tokens

            enriched_text = f"{chunk['content']}\n\n{contextualized_text}"

            return {
                # what gets embedded
                "text_to_embed": enriched_text,

                "metadata": {
                    # ── identification ────────────────────────────────────
                    "doc_id":         doc["doc_id"],
                    "original_uuid":  doc["original_uuid"],
                    "chunk_id":       chunk["chunk_id"],
                    "original_index": chunk["original_index"],

                    # ── inspectable content fields ────────────────────────
                    # kept separate so you can see exactly what Claude added
                    "original_content":       chunk["content"],
                    "contextualized_content": contextualized_text,

                    # ── enriched text stored in metadata ──────────────────
                    # PINECONE DIFFERENCE from ChromaDB:
                    # Pinecone has no separate "documents" field —
                    # only vectors + metadata exist. So we store the
                    # enriched text here if we want to retrieve it later.
                    "embedded_text": enriched_text,
                },

                "chunk_id": chunk["chunk_id"],
            }

        # ── Submit all chunks to thread pool ──────────────────────────────────
        texts_to_embed: list[str]            = []
        metadatas:      list[dict[str, Any]] = []
        ids:            list[str]            = []

        with ThreadPoolExecutor(max_workers=parallel_threads) as executor:
            futures = [
                executor.submit(process_chunk, doc, chunk)
                for doc in dataset
                for chunk in doc["chunks"]
            ]

            for future in tqdm(
                as_completed(futures),
                total=total_chunks,
                desc="Processing chunks",
            ):
                result = future.result()
                texts_to_embed.append(result["text_to_embed"])
                metadatas.append(result["metadata"])
                ids.append(result["chunk_id"])

        # ── Embed + store in Pinecone ─────────────────────────────────────────
        self._embed_and_store(texts_to_embed, metadatas, ids)

        # ── Token usage report ────────────────────────────────────────────────
        total_tokens = (
            self.token_counts["input"]
            + self.token_counts["cache_read"]
            + self.token_counts["cache_creation"]
        )
        savings_pct = (
            (self.token_counts["cache_read"] / total_tokens) * 100
            if total_tokens > 0
            else 0.0
        )

        print(f"\nContextual Pinecone DB loaded. Total chunks: {len(texts_to_embed):,}")
        print(f"  Input tokens (full price):    {self.token_counts['input']:,}")
        print(f"  Output tokens:                {self.token_counts['output']:,}")
        print(f"  Cache creation tokens:        {self.token_counts['cache_creation']:,}")
        print(f"  Cache read tokens (90% off):  {self.token_counts['cache_read']:,}")
        print(f"  Cache savings:                {savings_pct:.1f}% of input read from cache")

    # =========================================================================
    # _embed_and_store
    # =========================================================================

    def _embed_and_store(
        self,
        texts:     list[str],
        metadatas: list[dict[str, Any]],
        ids:       list[str],
    ) -> None:
        """
        Embed all texts in batches of 128 via Voyage AI, then upsert into
        Pinecone in batches of 100 (Pinecone's recommended upsert batch size).

        Two separate batch sizes:
            128 → Voyage API limit per embed call
            100 → Pinecone recommended upsert batch size

        Args:
            texts:     enriched strings (original + context) to embed
            metadatas: parallel list of metadata dicts (same order as texts)
            ids:       parallel list of chunk IDs  (same order as texts)
        """
        voyage_batch_size  = 128
        pinecone_batch_size = 100

        # ── Step 1: Voyage embedding in batches of 128 ────────────────────────
        print(f"Embedding {len(texts):,} chunks with Voyage AI...")

        raw_batches = [
            self.voyage_client.embed(
                texts[i : i + voyage_batch_size], model="voyage-4"
            ).embeddings
            for i in tqdm(
                range(0, len(texts), voyage_batch_size),
                desc="Embedding batches",
            )
        ]

        # flatten list-of-batches → flat list of vectors
        all_embeddings: list[list[float]] = [
            embedding
            for batch in raw_batches
            for embedding in batch
        ]

        # ── Step 2: Build Pinecone vector dicts ───────────────────────────────
        # Each vector is {"id": str, "values": list[float], "metadata": dict}
        vectors = [
            {
                "id":       ids[i],
                "values":   all_embeddings[i],
                "metadata": metadatas[i],
            }
            for i in range(len(ids))
        ]

        # ── Step 3: Upsert into Pinecone in batches of 100 ───────────────────
        # Pinecone recommends ≤100 vectors per upsert call for reliability
        print(f"Upserting {len(vectors):,} vectors into Pinecone...")

        for i in tqdm(
            range(0, len(vectors), pinecone_batch_size),
            desc="Upserting batches",
        ):
            batch = vectors[i : i + pinecone_batch_size]
            self.index.upsert(
                vectors   = batch,
                namespace = self.namespace,
            )

        print(
            f"Stored {len(vectors):,} vectors in Pinecone index "
            f"'{self.name}' / namespace '{self.namespace}'."
        )

    # =========================================================================
    # search
    # =========================================================================

    def search(
        self,
        query:         str,
        k:             int        = 20,
        doc_id_filter: str | None = None,
    ) -> list[dict[str, Any]]:
        """
        Retrieve the top-k most similar chunks for a query.

        Args:
            query:         The search string
            k:             Number of results to return
            doc_id_filter: Optional — restrict search to one document.
                           Pass a doc_id when debugging against ground truth
                           to isolate whether the problem is doc-level retrieval
                           or chunk-level ranking.

                           Two modes:
                             Production:  doc_id_filter=None
                               → searches entire namespace (all documents)
                             Debugging:   doc_id_filter="paper1"
                               → only ranks chunks from that document
                               → useful for comparing retrieved vs golden chunk

        Returns:
            List of dicts, each containing:
              - metadata    (original_content, contextualized_content, etc.)
              - embedded_text (the enriched text that was actually embedded)
              - score       (cosine similarity, higher = more similar, 0→1)
        """
        # ── Check index has data ──────────────────────────────────────────────
        stats  = self.index.describe_index_stats()
        counts = stats.get("namespaces", {})
        if not counts.get(self.namespace, {}).get("vector_count", 0):
            raise ValueError(
                f"Namespace '{self.namespace}' is empty. Run load_data() first."
            )

        # ── Query cache: skip Voyage call if seen before ──────────────────────
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.voyage_client.embed(
                [query], model="voyage-4"
            ).embeddings[0]
            self.query_cache[query] = query_embedding

        # ── Optional metadata pre-filter ─────────────────────────────────────
        # Pinecone filters BEFORE similarity search — much faster than
        # post-filtering in Python. Use doc_id for targeted debugging.
        #
        # Pinecone filter operators:
        #   $eq  → exact match       {"doc_id": {"$eq": "paper1"}}
        #   $in  → match any of list {"doc_id": {"$in": ["p1", "p2"]}}
        #   $gte → greater or equal  {"original_index": {"$gte": 5}}
        filter_dict = (
            {"doc_id": {"$eq": doc_id_filter}}
            if doc_id_filter is not None
            else None
        )

        # ── Pinecone similarity search ────────────────────────────────────────
        # PINECONE DIFFERENCE from ChromaDB:
        #   ChromaDB returns "distances" (lower = more similar)
        #   Pinecone returns "score"     (higher = more similar, already 0→1)
        #   No conversion needed here — score IS the similarity directly.
        raw = self.index.query(
            vector          = query_embedding,
            top_k           = k,
            namespace       = self.namespace,
            filter          = filter_dict,
            include_metadata= True,    # return metadata dict per result
            include_values  = False,   # skip returning raw vectors (saves bandwidth)
        )

        # ── Format results ────────────────────────────────────────────────────
        # raw["matches"] is a list of Match objects:
        # [{"id": ..., "score": ..., "metadata": {...}}, ...]
        results = []
        for match in raw["matches"]:
            results.append({
                "metadata":     match["metadata"],
                "embedded_text": match["metadata"].get("embedded_text", ""),
                # Pinecone score = cosine similarity directly (no conversion)
                "score": round(match["score"], 4),
            })

        return results


# =============================================================================
# Convenience: evaluate_db compatible wrapper
# =============================================================================

def retrieve_contextual_pinecone(
    query: str,
    db:    ContextualPineconeDB,
    k:     int = 20,
) -> list[dict[str, Any]]:
    """
    Drop-in replacement for retrieve_base() in evaluate_retrieval().
    Wraps ContextualPineconeDB.search() to match the expected signature.
    """
    return db.search(query, k=k)



In [ ]:

# =============================================================================
# Main — quick smoke test
# =============================================================================

if __name__ == "__main__":
    # ── Load dataset produced by DocumentProcessor ────────────────────────────
    with open("./data/codebase_chunks.json", encoding="utf-8") as f:
        dataset = json.load(f)

    # ── Build contextual Pinecone DB ──────────────────────────────────────────
    db = ContextualPineconeDB(
        name      = "rag-research",
        namespace = "research_papers",
        use_pod   = False,   # serverless for dev; set True for production
    )

    db.load_data(dataset, parallel_threads=1)

    # ── Quick search smoke test ───────────────────────────────────────────────
    results = db.search("how does DPR handle out-of-domain retrieval?", k=5)

    print("\n── Top 5 results ──────────────────────────────────────────────")
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] score={r['score']:.4f}  chunk={r['metadata']['chunk_id']}")
        print(f"     original:      {r['metadata']['original_content'][:120]}...")
        print(f"     contextualized:{r['metadata']['contextualized_content'][:120]}...")

    # ── Debug: search within a specific document ──────────────────────────────
    print("\n── Filtered search (doc_id='paper1') ──────────────────────────")
    filtered = db.search(
        "how does DPR handle out-of-domain retrieval?",
        k=5,
        doc_id_filter="paper1",
    )
    for i, r in enumerate(filtered, 1):
        print(f"[{i}] score={r['score']:.4f}  chunk={r['metadata']['chunk_id']}")